# Qwen 2.5 7B - A1/A2 Logit Lens (digit-split aware)

**Runtime -> T4 GPU, then Run all.**

Cell 2 restarts the session (expected) - after it reconnects, hit Run all again.

Qwen splits numbers into single digits, so the lens reads every digit position and reconstructs the full number. Same experiment as Llama, readout adapted to Qwen's tokenizer.

**Watch:** Cell 6 (tokenizer), Cell 8 (reconstruction test), first 2 problems of the run.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 27.2 MB/s eta 0:00:00


In [ ]:
# Clean restart so bitsandbytes loads. Session drops and reconnects - expected.
import os
os.kill(os.getpid(), 9)

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID   = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MODEL_NAME = "qwen_2.5_7b"
N_LAYERS   = 28

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"": 0})
model.eval()
torch.cuda.empty_cache()

print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"REAL layers: {model.config.num_hidden_layers}  |  N_LAYERS: {N_LAYERS}")
assert model.config.num_hidden_layers == N_LAYERS, "LAYER MISMATCH - fix N_LAYERS"
assert hasattr(model.model, "norm"), "model.model.norm missing"
assert hasattr(model, "lm_head"), "model.lm_head missing"
print("architecture OK")

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

GPU memory: 5.55 GB
REAL layers: 28  |  N_LAYERS: 28
architecture OK


In [2]:
import os, json, subprocess
import numpy as np

if not os.path.exists("/content/RuleArena"):
    subprocess.run(["git","clone","-q",
        "https://github.com/SkyRiver-2000/RuleArena.git","/content/RuleArena"], check=True)

os.chdir("/content/RuleArena/airline")
exec(open("compute_answer.py").read().split("if __name__")[0])
check_base_tables = load_checking_fee()

problems = []
with open("synthesized_problems/comp_0.jsonl") as f:
    for line in f:
        problems.append(json.loads(line))

def compute_gt(info):
    total, gt_info = compute_answer(
        base_price=info["base_price"], direction=info["direction"],
        routine=info["routine"], customer_class=info["customer_class"],
        bag_list=[{"id":0,"name":"ticket","size":[0,0,0],"weight":0}] + info["bag_list"],
        check_base_tables=check_base_tables)
    return total, gt_info

print("Problems:", len(problems), "| GT problem 0:", compute_gt(problems[0]["info"])[0])

Problems: 100 | GT problem 0: 1445


In [3]:
import re

with open("/content/RuleArena/airline/reference_rules.txt") as f:
    REFERENCE_RULES = f.read()
print(f"Rules loaded: {len(REFERENCE_RULES)} chars")


def generate_steps(problem, gt_info):
    info = problem["info"]
    steps = []

    for i, bag in enumerate(info["bag_list"]):
        bag_id = bag["id"]
        dims = bag["size"]
        weight = bag["weight"]
        dim_sum = sum(dims)

        correct_oversize_fee = gt_info["oversize"][i]
        correct_overweight_fee = gt_info["overweight"][i]
        correct_base_fee = gt_info["base"][i]

        steps.append({
            "step_id": f"bag{bag_id}_EA",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compute dimension sum",
            "system": "You are a calculator. Answer with just the number.",
            "user": f"What is {dims[0]} + {dims[1]} + {dims[2]}? Answer with just the number.",
            "correct_answer": str(dim_sum),
        })

        expected_oversize = "OVERSIZE" if dim_sum > 62 else "NOT OVERSIZE"
        steps.append({
            "step_id": f"bag{bag_id}_EB",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare sum to oversize threshold",
            "system": "Answer with exactly OVERSIZE or NOT OVERSIZE.",
            "user": (
                f"A bag has total dimensions of {dim_sum} inches. "
                f"The oversize threshold is 62 inches. "
                f"Is this bag OVERSIZE or NOT OVERSIZE?"
            ),
            "correct_answer": expected_oversize,
        })

        expected_overweight = "OVERWEIGHT" if weight > 50 else "NOT OVERWEIGHT"
        steps.append({
            "step_id": f"bag{bag_id}_EC",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare weight to overweight threshold",
            "system": "Answer with exactly OVERWEIGHT or NOT OVERWEIGHT.",
            "user": (
                f"A bag weighs {weight} lbs. "
                f"The overweight threshold is 50 lbs. "
                f"Is this bag OVERWEIGHT or NOT OVERWEIGHT?"
            ),
            "correct_answer": expected_overweight,
        })

        if dim_sum > 62:
            if dim_sum <= 65:
                bracket = "Over 62 inches to 65 inches"
            else:
                bracket = "Over 65 inches to 115 inches"
            steps.append({
                "step_id": f"bag{bag_id}_ED",
                "bag_id": bag_id,
                "type": "elementary" if correct_oversize_fee > 0 else "boundary",
                "description": "Read oversize fee from table given bracket"
                               + (" (complimentary rule applies -> boundary)" if correct_oversize_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {bracket}.\n\n"
                    f"OVERSIZE FEE TABLE:\n"
                    f"Between U.S., Puerto Rico, U.S. Virgin Islands and Canada:\n"
                    f"- Over 62 inches to 65 inches: $30\n"
                    f"- Over 65 inches to 115 inches: $200\n\n"
                    f"What is the oversize fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            if weight <= 53:
                wbracket = "Over 50 lbs to 53 lbs"
            elif weight <= 70:
                wbracket = "Over 53 lbs to 70 lbs"
            else:
                wbracket = "Over 70 lbs to 100 lbs"
            steps.append({
                "step_id": f"bag{bag_id}_EE",
                "bag_id": bag_id,
                "type": "elementary" if correct_overweight_fee > 0 else "boundary",
                "description": "Read overweight fee from table given bracket"
                               + (" (complimentary rule applies -> boundary)" if correct_overweight_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {wbracket}.\n\n"
                    f"OVERWEIGHT FEE TABLE:\n"
                    f"- Over 50 lbs to 53 lbs: $30\n"
                    f"- Over 53 lbs to 70 lbs: $100\n"
                    f"- Over 70 lbs to 100 lbs: $200\n\n"
                    f"What is the overweight fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

        steps.append({
            "step_id": f"bag{bag_id}_B0",
            "bag_id": bag_id,
            "type": "boundary",
            "description": "Is bag complimentary? (class + bag number + route rule)",
            "system": "Answer with exactly YES or NO.",
            "user": (
                f"Passenger class: {gt_info['customer_class']}\n"
                f"Route: {gt_info['routine']} domestic\n"
                f"This is bag number {bag_id} "
                f"(the {['1st','2nd','3rd','4th','5th'][bag_id-1]} checked bag).\n\n"
                f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                f"Is this bag complimentary (free) for this passenger? "
                f"Answer with exactly YES or NO."
            ),
            "correct_answer": "YES" if correct_base_fee == 0 else "NO",
        })

        if dim_sum > 62:
            steps.append({
                "step_id": f"bag{bag_id}_B1",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw dimensions -> oversize fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag dimensions: {dims[0]} x {dims[1]} x {dims[2]} inches.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the oversize fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            steps.append({
                "step_id": f"bag{bag_id}_B2",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw weight -> overweight fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag weight: {weight} lbs.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the overweight fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

    return steps


def extract_answer(response, step_type):
    response = response.strip()
    for keyword in ["NOT OVERSIZE", "NOT OVERWEIGHT", "OVERSIZE", "OVERWEIGHT"]:
        if keyword in response.upper():
            return keyword
    if response.upper().startswith("YES"):
        return "YES"
    if response.upper().startswith("NO"):
        return "NO"
    match = re.search(r'\$(\d+)', response)
    if match:
        return f"${match.group(1)}"
    nums = re.findall(r'\b\d+\b', response)
    if nums:
        return nums[0]
    return response.strip()


def build_prompt(step):
    messages = [
        {"role": "system", "content": step["system"]},
        {"role": "user", "content": step["user"]},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


gt, gt_info = compute_gt(problems[0]["info"])
_steps = generate_steps(problems[0], gt_info)
print(f"Problem 0: {len(_steps)} steps - "
      f"{sum(1 for s in _steps if s['type']=='elementary')} elementary, "
      f"{sum(1 for s in _steps if s['type']=='boundary')} boundary")

Rules loaded: 19583 chars
Problem 0: 36 steps - 23 elementary, 13 boundary


In [4]:
print("How Qwen tokenizes answers:\n")
for s in ["$30", "$100", "$200", "86", "OVERSIZE", "YES"]:
    ids = tokenizer(s, add_special_tokens=False)["input_ids"]
    print(f"  {s!r:12} -> {[tokenizer.decode([i]) for i in ids]}")

How Qwen tokenizes answers:

  '$30'        -> ['$', '3', '0']
  '$100'       -> ['$', '1', '0', '0']
  '$200'       -> ['$', '2', '0', '0']
  '86'         -> ['8', '6']
  'OVERSIZE'   -> ['OVER', 'SIZE']
  'YES'        -> ['YES']


In [5]:
import torch
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

@torch.no_grad()
def answer_and_lens_fast(step, max_new_tokens=8):
    prompt = build_prompt(step)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = enc["input_ids"].shape[-1]

    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        output_hidden_states=True, return_dict_in_generate=True,
    )
    answer_ids = out.sequences[0][prompt_len:]
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
    ans_tokens = [tokenizer.decode([t]) for t in answer_ids]

    # find the full run of digit tokens = the whole number
    digit_positions = []          # list of (generation_index, token_id, digit_str)
    started = False
    for j, t in enumerate(ans_tokens):
        ts = t.strip()
        is_digit = len(ts) > 0 and ts.replace(",", "").isdigit()
        if is_digit:
            digit_positions.append((j, answer_ids[j].item(), ts))
            started = True
        elif started:
            break

    norm, head = model.model.norm, model.lm_head

    def lens_at(gen_idx, target_id):
        """logit lens at one generated position for one target token"""
        step_hs = out.hidden_states[gen_idx]
        c_top1, c_p90, probs = None, None, []
        top5 = None
        for L in range(1, N_LAYERS + 1):
            h = step_hs[L][0, -1]
            p = torch.softmax(head(norm(h)).float(), dim=-1)
            pt = p[target_id].item()
            probs.append(round(pt, 4))
            if c_top1 is None and torch.argmax(p).item() == target_id:
                c_top1 = L
            if c_p90 is None and pt > 0.9:
                c_p90 = L
            if L == N_LAYERS:
                v, i = torch.topk(p, 5)
                top5 = [(tokenizer.decode([a]), round(float(b), 4)) for a, b in zip(i.tolist(), v.tolist())]
        return (c_top1 or N_LAYERS), (c_p90 or N_LAYERS), probs, top5

    if not digit_positions:
        # non-numeric answer (YES/NO/OVERSIZE) -> read first generated token
        tid = answer_ids[0].item()
        c1, c9, probs, top5 = lens_at(0, tid)
        return {
            "answer_text": answer_text,
            "reconstructed_number": ans_tokens[0].strip(),
            "is_numeric": False,
            "commit_top1": c1,
            "commit_p90": c9,
            "per_digit": None,
            "probs_by_layer": probs,
            "top5_final": top5,          # first-position top5
        }

    # numeric answer -> read every digit, reconstruct
    per_digit = []
    for (j, tid, tstr) in digit_positions:
        c1, c9, probs, top5 = lens_at(j, tid)
        per_digit.append({
            "digit": tstr,
            "commit_top1": c1,
            "commit_p90": c9,
            "probs_by_layer": probs,
            "top5_final": top5,
        })

    reconstructed = "".join(d["digit"] for d in per_digit)
    # full-number commit = slowest digit to commit (number not decided until last digit settles)
    commit_top1 = max(d["commit_top1"] for d in per_digit)
    commit_p90  = max(d["commit_p90"]  for d in per_digit)

    return {
        "answer_text": answer_text,
        "reconstructed_number": reconstructed,
        "is_numeric": True,
        "commit_top1": commit_top1,
        "commit_p90": commit_p90,
        "per_digit": per_digit,          # full per-digit detail for A2
        "top5_final": per_digit[0]["top5_final"],   # first-digit top5, for reference
        "probs_by_layer": per_digit[0]["probs_by_layer"],
    }

print("lens ready - reads all digit positions, reconstructs full number")

lens ready - reads all digit positions, reconstructs full number


In [6]:
gt, gt_info = compute_gt(problems[0]["info"])
for sid in ["bag2_EA", "bag2_ED", "bag2_B2"]:
    s = next(x for x in generate_steps(problems[0], gt_info) if x["step_id"]==sid)
    r = answer_and_lens_fast(s)
    print(f"{sid} [{s['type']}]")
    print(f"   correct   : {s['correct_answer']}")
    print(f"   answer    : {r['answer_text'].strip()!r}")
    print(f"   reconstru.: {r['reconstructed_number']!r}")
    print(f"   commit    : {r['commit_top1']}")
    if r["per_digit"]:
        for d in r["per_digit"]:
            print(f"      digit {d['digit']}: commit L{d['commit_top1']}")
    print()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


bag2_EA [elementary]
   correct   : 86
   answer    : '86'
   reconstru.: '86'
   commit    : 27
      digit 8: commit L27
      digit 6: commit L27

bag2_ED [elementary]
   correct   : $200
   answer    : '200'
   reconstru.: '200'
   commit    : 27
      digit 2: commit L26
      digit 0: commit L27
      digit 0: commit L8

bag2_B2 [boundary]
   correct   : $100
   answer    : '100'
   reconstru.: '100'
   commit    : 26
      digit 1: commit L26
      digit 0: commit L26
      digit 0: commit L24



In [7]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/derivation_boundaries"
os.makedirs(SAVE_DIR, exist_ok=True)
RESULTS_PATH_QWEN = f"{SAVE_DIR}/A1_A2_logit_lens_clean_qwen.json"
print("Saving to:", RESULTS_PATH_QWEN)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/derivation_boundaries/A1_A2_logit_lens_clean_qwen.json


In [8]:
import time

START_IDX = 0
END_IDX = 30      # full run, all 60 problems

if os.path.exists(RESULTS_PATH_QWEN):
    all_results = json.load(open(RESULTS_PATH_QWEN))
    done = {r["problem_idx"] for r in all_results}
    print(f"Resuming - {len(done)} problems, {len(all_results)} steps.")
else:
    all_results, done = [], set()
    print("Starting fresh.")

t_start = time.time()

for idx in range(START_IDX, END_IDX):
    if idx in done:
        continue
    try:
        gt, gt_info = compute_gt(problems[idx]["info"])
        steps = generate_steps(problems[idx], gt_info)

        for s in steps:
            r = answer_and_lens_fast(s)
            pred = extract_answer(r["answer_text"], s["type"])
            all_results.append({
                "problem_idx": idx, "step_id": s["step_id"], "type": s["type"],
                "description": s["description"], "correct_answer": s["correct_answer"],
                "predicted": pred,
                "correct": pred.strip().lstrip("$").upper() == s["correct_answer"].strip().lstrip("$").upper(),
                "reconstructed_number": r["reconstructed_number"],
                "is_numeric": r["is_numeric"],
                "commit_top1": r["commit_top1"], "commit_p90": r["commit_p90"],
                "per_digit": r["per_digit"],
                "top5_final": r["top5_final"],
                "probs_by_layer": r["probs_by_layer"],
            })

        with open(RESULTS_PATH_QWEN, "w") as f:
            json.dump(all_results, f)

        def sfx(x): return x["step_id"].split("_")[1]
        giv = [x for x in all_results if sfx(x) in ("ED","EE")]
        der = [x for x in all_results if sfx(x) in ("B1","B2")]
        mins = (time.time() - t_start) / 60
        n_done = len([i for i in range(START_IDX, idx+1) if i not in done])
        eta = mins / max(n_done, 1) * (END_IDX - idx - 1)
        print(f"[{idx+1}/{END_IDX}] steps={len(all_results)}  "
              f"GIVEN acc={np.mean([x['correct'] for x in giv]):.1%} L={np.mean([x['commit_top1'] for x in giv]):.2f}  |  "
              f"DERIVED acc={np.mean([x['correct'] for x in der]):.1%} L={np.mean([x['commit_top1'] for x in der]):.2f}  "
              f"({mins:.0f}m, ~{eta:.0f}m left)")

    except Exception as e:
        print(f"  ERROR problem {idx}: {e}")
        torch.cuda.empty_cache()
        continue

print(f"\nDone. {len(all_results)} steps, {(time.time()-t_start)/60:.0f} min.")

Starting fresh.
[1/30] steps=36  GIVEN acc=100.0% L=27.00  |  DERIVED acc=37.5% L=25.50  (7m, ~202m left)
[2/30] steps=72  GIVEN acc=93.8% L=27.00  |  DERIVED acc=37.5% L=25.62  (14m, ~195m left)
[3/30] steps=108  GIVEN acc=91.7% L=27.00  |  DERIVED acc=33.3% L=25.58  (21m, ~188m left)
[4/30] steps=144  GIVEN acc=93.8% L=27.00  |  DERIVED acc=31.2% L=25.56  (28m, ~181m left)
[5/30] steps=180  GIVEN acc=85.0% L=27.00  |  DERIVED acc=30.0% L=25.57  (35m, ~174m left)
[6/30] steps=216  GIVEN acc=87.5% L=27.00  |  DERIVED acc=33.3% L=25.56  (42m, ~167m left)
[7/30] steps=252  GIVEN acc=85.7% L=27.00  |  DERIVED acc=33.9% L=25.55  (49m, ~160m left)
[8/30] steps=288  GIVEN acc=84.4% L=27.00  |  DERIVED acc=29.7% L=25.55  (56m, ~153m left)
[9/30] steps=324  GIVEN acc=86.1% L=27.00  |  DERIVED acc=30.6% L=25.54  (63m, ~146m left)
[10/30] steps=360  GIVEN acc=87.5% L=27.00  |  DERIVED acc=32.5% L=25.54  (70m, ~139m left)
[11/30] steps=396  GIVEN acc=88.6% L=27.00  |  DERIVED acc=34.1% L=25.53  (